In [1]:
!pip install bert-score sentence-transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [1]:
import pandas as pd
import torch
from tqdm import tqdm
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv('/content/qna-clean.csv')

# Initialize SBERT and move to GPU
sbert = SentenceTransformer('all-MiniLM-L6-v2')
sbert = sbert.to('cuda' if torch.cuda.is_available() else 'cpu')

# Output lists
cos_sims = []
bert_score_f1s = []

# Set batch size
batch_size = 32

# Batching loop
for i in tqdm(range(0, len(df), batch_size), desc="Processing batches"):
    topics = df['MAIN'].iloc[i:i+batch_size].tolist()
    comments = df['comment_body'].iloc[i:i+batch_size].tolist()

    # SBERT embeddings for cosine similarity
    topic_embs = sbert.encode(topics, convert_to_tensor=True, device='cuda')
    comment_embs = sbert.encode(comments, convert_to_tensor=True, device='cuda')

    # Cosine similarity
    cos_batch = torch.nn.functional.cosine_similarity(topic_embs, comment_embs).tolist()
    cos_sims.extend(cos_batch)

    # BERTScore (batched and on GPU)
    _, _, f1 = bert_score(comments, topics, lang='en', verbose=False, device='cuda', rescale_with_baseline=True)
    bert_score_f1s.extend(f1.tolist())

# Add to DataFrame
df['cos_sim'] = cos_sims
df['bert_score'] = bert_score_f1s

# Save to new CSV
df.to_csv('qna_with_features.csv', index=False)
print("✅ Features saved to 'qna_with_features.csv'")

ModuleNotFoundError: No module named 'bert_score'

In [3]:
import pandas as pd
import torch
from tqdm import tqdm
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv('/content/CRYPTO_QnA_TEST.csv')

# Initialize SBERT and move to GPU
sbert = SentenceTransformer('all-MiniLM-L6-v2')
sbert = sbert.to('cuda' if torch.cuda.is_available() else 'cpu')

# Output lists
cos_sims = []
bert_score_f1s = []

# Set batch size
batch_size = 32

# Batching loop
for i in tqdm(range(0, len(df), batch_size), desc="Processing batches"):
    topics = df['MAIN'].iloc[i:i+batch_size].tolist()
    comments = df['comment_body'].iloc[i:i+batch_size].tolist()

    # SBERT embeddings for cosine similarity
    topic_embs = sbert.encode(topics, convert_to_tensor=True, device='cuda')
    comment_embs = sbert.encode(comments, convert_to_tensor=True, device='cuda')

    # Cosine similarity
    cos_batch = torch.nn.functional.cosine_similarity(topic_embs, comment_embs).tolist()
    cos_sims.extend(cos_batch)

    # BERTScore (batched and on GPU)
    _, _, f1 = bert_score(comments, topics, lang='en', verbose=False, device='cuda', rescale_with_baseline=True)
    bert_score_f1s.extend(f1.tolist())

# Add to DataFrame
df['cos_sim'] = cos_sims
df['bert_score'] = bert_score_f1s

# Save to new CSV
df.to_csv('CRYPTO_QnA_TEST_with_features.csv', index=False)
print("✅ Features saved to 'CRYPTO_QnA_TEST_with_features.csv'")

VADER lexicon not found, attempting to download...
VADER lexicon downloaded.


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


Dataset loaded successfully.

Performing VADER sentiment analysis on the 'MAIN' column...

VADER sentiment analysis complete.
                                               title selftext  \
0  NOOB ALERT. Some insight (DNT)? Is it about to...      NaN   
1    Is adding name/address necessary to buy crypto?      NaN   
2                         https://t.me/Victory_Token      NaN   
3  Five years ago, the crypto community became aw...      NaN   
4                       Beginner in Crypto Investing      NaN   

                                                MAIN  Level 1  Level 2  \
0  noob alert. some insight (dnt)? is it about to...        2      0.0   
1   is adding name/address necessary to buy crypto?         2      0.0   
2                        https://t.me/Victory_Token         0      NaN   
3  Five years ago, the crypto community became aw...        1      NaN   
4                      beginner in crypto investing         2      0.0   

   Level 3  senti_score  
0      1.0  